In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
##Load the dataset
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
## Preprocessing the data
## Drop Irrerelevant columns
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)    ## axis = 1 means drop columns, axis = 0 means drop rows
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
## Encode categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])     ##fit_transform() method is used to fit the label encoder and transform the data in one step. It returns an array of encoded labels.
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [8]:
## One Hot Encoding for 'Geography' column
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(data[['Geography']])  ## fit_transform() method is used to fit the one hot encoder and transform the data in one step. It returns an array of encoded labels.
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [10]:
onehot_encoder_geo.get_feature_names_out(['Geography'])     ## get_feature_names_out() method is used to get the feature names of the one hot encoder. It returns an array of feature names.

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [12]:
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))    ## pd.DataFrame() method is used to create a DataFrame from the encoded labels. It returns a DataFrame.
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [13]:
## Combine one hot encoded columns with the original dataset
data = pd.concat([data.drop(['Geography'], axis=1), geo_encoded_df], axis=1)    ## pd.concat() method is used to concatenate two DataFrames. It returns a DataFrame.
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [14]:
## Save the encoders and scaler for future use
with open('laber_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

In [15]:
## Divide the dataset into independent and dependent features
X = data.drop(['Exited'], axis=1)    ## X is the independent features, drop the target column 'Exited'
y = data['Exited']    ## y is the dependent feature, the target column 'Exited'

## Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)    ## test_size=0.2 means 20% of the data will be used for testing, random_state=42 is used to get the same split every time

## Scale the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)    ## fit_transform() method is used to fit the scaler and transform the data in one step. It returns an array of scaled features.
X_test = scaler.transform(X_test)    ## transform() method is used to transform the data using the already fitted scaler. It returns an array of scaled features.


In [16]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]], shape=(8000, 12))

In [17]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [18]:
## ANN Implementation

In [19]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [21]:
(X_train.shape[1],)    #input_shape is the number of features in the dataset

(12,)

In [22]:
## Build the ANN model
model = Sequential([
    Dense(64,activation='relu', input_shape=(X_train.shape[1],)),   ## HL1 connected with input layer    ## input_shape is the number of features in the dataset
    Dense(32,activation='relu'),   ## HL2
    Dense(1,activation='sigmoid') ## Output layer
])


c:\ANN_Project\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [23]:
model.summary()    ## summary() method is used to print the summary of the model. It returns a summary of the model.

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)    ## Adam optimizer is used to optimize the model. It takes learning_rate as an argument. It returns an optimizer object.
loss = tensorflow.keras.losses.BinaryCrossentropy()    ## BinaryCrossentropy loss function is used to calculate the loss of the model. It returns a loss object.
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x000002799465E8E0>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

In [26]:
## Compile the model
model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])    ## compile() method is used to compile the model. It takes three arguments: optimizer, loss function and metrics. It returns None.

In [27]:
## Setup the Tensorboard
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")    ## log_dir is the directory where the logs will be saved. It takes the current date and time as the name of the directory.
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)    ## TensorBoard() method is used to create a TensorBoard callback. It takes log_dir and histogram_freq as arguments. It returns a TensorBoard callback object.

In [30]:
## Setup EarlyStopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)    ## EarlyStopping() method is used to create an EarlyStopping callback. It takes monitor, patience and restore_best_weights as arguments. It returns an EarlyStopping callback object. 


In [31]:
### Train the model
history = model.fit(X_train, y_train, validation_data=(X_test,y_test), epochs=100, 
                    callbacks=[tensorflow_callback, early_stopping_callback])       
                    ## fit() method is used to train the model. It takes X_train, y_train, epochs, batch_size, validation_split and callbacks as arguments. It returns a History object.

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8580 - loss: 0.3417 - val_accuracy: 0.8585 - val_loss: 0.3430
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8616 - loss: 0.3374 - val_accuracy: 0.8505 - val_loss: 0.3472
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8614 - loss: 0.3354 - val_accuracy: 0.8580 - val_loss: 0.3415
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8595 - loss: 0.3344 - val_accuracy: 0.8620 - val_loss: 0.3385
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8634 - loss: 0.3322 - val_accuracy: 0.8580 - val_loss: 0.3383
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8636 - loss: 0.3284 - val_accuracy: 0.8610 - val_loss: 0.3426
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8666 - loss: 0.3273 - val_accuracy: 0.8605 - val_loss: 0.3484
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8659 - loss: 0.3270 - val_accu

In [32]:
model.save('model.h5')    ## save() method is used to save the model. It takes the name of the file as an argument. It returns None.

In [37]:
## Load Tensorboard Extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [38]:
%tensorboard --logdir logs/fit/20260721-200830

Reusing TensorBoard on port 6007 (pid 17856), started 0:01:37 ago. (Use '!kill 17856' to kill it.)

In [ ]:
## Load the pickle file
